In [ ]:
import pymupdf4llm
import pathlib
import pdfplumber

# Define paths
pdf_path = "textbook.pdf"
output_path = "textbook_clean.md"

# 1. Convert the entire PDF directly to Markdown syntax
print("Converting PDF to Markdown...")
md_text = pymupdf4llm.to_markdown(pdf_path)

# 2. Save the output
pathlib.Path(output_path).write_bytes(md_text.encode("utf-8"))
print(f"Done! Saved to {output_path}")



pdf_path = "textbook.pdf"
md_path = "textbook_raw.md"

with pdfplumber.open(pdf_path) as pdf:
    with open(md_path, "w", encoding="utf-8") as md_file:
        for page_num, page in enumerate(pdf.pages, start=1):
            text = page.extract_text()

            if text:
                # Add a markdown header for the page tracking (great for chunking later)
                md_file.write(f"\n\n\n\n")
                md_file.write(text)

print("Raw text extracted!")

ModuleNotFoundError: No module named 'pymupdf4llm'

In [ ]:
import google.generativeai as genai
from google.colab import userdata
import json
import os
import time

# 1. Setup
markdown_file_path = "/content/thermo.md"
if not os.path.exists(markdown_file_path):
    print(f"❌ Error: {markdown_file_path} not found!")
else:
    with open(markdown_file_path, 'r', encoding='utf-8') as f:
        full_text_md = f.read()

    api_key = userdata.get('Gemin_Key')
    genai.configure(api_key=api_key)

    # UPDATED MODEL NAMES (Stable 2026 versions)
    # Using 'gemini-2.0-pro' for creation and 'gemini-2.0-flash' for auditing
    pro_model = genai.GenerativeModel('gemini-2.0-pro')
    flash_model = genai.GenerativeModel('gemini-2.0-flash')

    # 2. Chunking
    chunks = [full_text_md[i:i+2800] for i in range(0, len(full_text_md), 2800)]

    # --- STEP 1: INSTRUCTION GENERATION (PRO) ---
    def generate_instruction(chunk):
        prompt = f"""You are a Professor of Chemical Engineering.
Based on the technical data below, generate ONE complex Thermodynamics prompt.

CRITICAL CONSTRAINTS:
- FORMAT: This must be a standalone "instruction" (a question or challenge).
- CONTENT: Can be a quantitative calculation OR a deep conceptual reasoning challenge.
- NO META-TALK: DO NOT mention "the excerpt," "the text," or "the provided data."
- RETURN: Provide ONLY a JSON object with a single key: "instruction".

TECHNICAL DATA: {chunk}"""
        try:
            # Setting response_mime_type ensures the model adheres to JSON schema
            response = pro_model.generate_content(
                prompt,
                generation_config={"response_mime_type": "application/json"}
            )
            return json.loads(response.text).get("instruction")
        except Exception as e:
            print(f"⚠️ Step 1 Error: {e}")
            return None

    # --- STEP 2: OUTPUT GENERATION (PRO) ---
    def generate_output(instruction, chunk):
        prompt = f"""You are a Thermodynamics Specialist. Provide a rigorous response to the instruction below.

INSTRUCTION: {instruction}
CONTEXT: {chunk}

Guidelines:
- MATH: Use LaTeX for all formulas (e.g., $PV^n = C$).
- JSON SAFETY: You MUST double-escape all backslashes (e.g., \\\\Delta, \\\\frac{{1}}{{2}}).
- REASONING: Provide a step-by-step logical derivation or calculation.
- FORMAT: Return a JSON object with keys "instruction" and "output".
"""
        try:
            response = pro_model.generate_content(
                prompt,
                generation_config={"response_mime_type": "application/json"}
            )
            return json.loads(response.text)
        except Exception as e:
            print(f"⚠️ Step 2 Error: {e}")
            return None

    # --- STEP 3: VETTING (FLASH) ---
    def vet_data(pair, chunk):
        prompt = f"""You are a Technical Auditor. Review this data for scientific accuracy.

SOURCE TEXT: {chunk}
PROPOSED DATA: {json.dumps(pair)}

Criteria:
1. Is the "output" scientifically sound based on the source?
2. Does the "instruction" avoid mentioning the source text?
3. Are backslashes correctly double-escaped?

Respond ONLY with 'VALID' or 'INVALID'."""
        try:
            response = flash_model.generate_content(
                prompt,
                generation_config={"temperature": 0.1}
            )
            return "VALID" in response.text.upper()
        except:
            return False

    # 3. Execution Loop
    dataset_file = "thermo_vetted_data.jsonl"
    for i, chunk in enumerate(chunks):
        print(f"📊 Processing Batch {i+1}/{len(chunks)}...")

        # Step 1: Instruction
        instr = generate_instruction(chunk)
        if not instr: continue

        # Step 2: Full Pair
        pair = generate_output(instr, chunk)
        if not pair: continue

        # Step 3: Vet
        if vet_data(pair, chunk):
            with open(dataset_file, "a", encoding="utf-8") as f:
                f.write(json.dumps(pair) + "\n")
            print(f"   ✅ Batch {i+1} saved.")
        else:
            print(f"   🗑️ Batch {i+1} rejected.")

        time.sleep(2)

    print("\n🎉 MISSION COMPLETE!")

❌ Error: /content/thermo.md not found!


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
